# Welcome to Week 2!

## Subtitle: Connecting to Multiple LLM APIs
This notebook demonstrates how to connect to and use multiple frontier LLM APIs including OpenAI, Anthropic Claude, and Google Gemini.
We explore different model capabilities, streaming responses, and create conversational interactions between different AI models.
The examples progress from simple API calls to more complex multi-model conversations and streaming implementations.

## Frontier Model APIs

In Week 1, we used multiple Frontier LLMs through their Chat UI, and we connected with the OpenAI's API.

Today we'll connect with the APIs for Anthropic and Google, as well as OpenAI.

<table style="margin: 0; text-align: left;">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../important.jpg" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#900;">Important Note - Please read me</h2>
            <span style="color:#900;">I'm continually improving these labs, adding more examples and exercises.
            At the start of each week, it's worth checking you have the latest code.<br/>
            First do a <a href="https://chatgpt.com/share/6734e705-3270-8012-a074-421661af6ba9">git pull and merge your changes as needed</a>. Any problems? Try asking ChatGPT to clarify how to merge - or contact me!<br/><br/>
            After you've pulled the code, from the llm_engineering directory, in an Anaconda prompt (PC) or Terminal (Mac), run:<br/>
            <code>conda env update --f environment.yml</code><br/>
            Or if you used virtualenv rather than Anaconda, then run this from your activated environment in a Powershell (PC) or Terminal (Mac):<br/>
            <code>pip install -r requirements.txt</code>
            <br/>Then restart the kernel (Kernel menu >> Restart Kernel and Clear Outputs Of All Cells) to pick up the changes.
            </span>
        </td>
    </tr>
</table>
<table style="margin: 0; text-align: left;">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../resources.jpg" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#f71;">Reminder about the resources page</h2>
            <span style="color:#f71;">Here's a link to resources for the course. This includes links to all the slides.<br/>
            <a href="https://edwarddonner.com/2024/11/13/llm-engineering-resources/">https://edwarddonner.com/2024/11/13/llm-engineering-resources/</a><br/>
            Please keep this bookmarked, and I'll continue to add more useful links there over time.
            </span>
        </td>
    </tr>
</table>

## Setting up your keys

If you haven't done so already, you could now create API keys for Anthropic and Google in addition to OpenAI.

**Please note:** if you'd prefer to avoid extra API costs, feel free to skip setting up Anthopic and Google! You can see me do it, and focus on OpenAI for the course. You could also substitute Anthropic and/or Google for Ollama, using the exercise you did in week 1.

For OpenAI, visit https://openai.com/api/  
For Anthropic, visit https://console.anthropic.com/  
For Google, visit https://ai.google.dev/gemini-api  

### Also - adding DeepSeek if you wish

Optionally, if you'd like to also use DeepSeek, create an account [here](https://platform.deepseek.com/), create a key [here](https://platform.deepseek.com/api_keys) and top up with at least the minimum $2 [here](https://platform.deepseek.com/top_up).

### Adding API keys to your .env file

When you get your API keys, you need to set them as environment variables by adding them to your `.env` file.

```
OPENAI_API_KEY=xxxx
ANTHROPIC_API_KEY=xxxx
GOOGLE_API_KEY=xxxx
DEEPSEEK_API_KEY=xxxx
```

Afterwards, you may need to restart the Jupyter Lab Kernel (the Python process that sits behind this notebook) via the Kernel menu, and then rerun the cells from the top.

In [ ]:
# Import required libraries for API connections and environment management
# This cell sets up all the necessary imports for connecting to multiple LLM APIs
# We'll use these libraries to manage API keys, make requests, and display results

import os
import time
import anthropic
from google import genai
from google.genai import types
from openai import OpenAI
from dotenv import load_dotenv

from IPython.display import Markdown, display, update_display

In [ ]:
# Load API keys from environment variables stored in .env file
# This cell securely loads API credentials and validates their presence
# Print the key prefixes to help with any debugging without exposing full keys

load_dotenv(override=True)
openai_api_key = os.getenv('OPENAI_API_KEY')
anthropic_api_key = os.getenv('ANTHROPIC_API_KEY')
google_api_key = os.getenv('GEMINI_API_KEY')

# Check if each API key is properly loaded and display partial key for verification
if openai_api_key:
    print(f"OpenAI API Key exists and begins {openai_api_key[:8]}")
else:
    print("OpenAI API Key not set")
    
if anthropic_api_key:
    print(f"Anthropic API Key exists and begins {anthropic_api_key[:7]}")
else:
    print("Anthropic API Key not set")

if google_api_key:
    print(f"Google API Key exists and begins {google_api_key[:8]}")
else:
    print("Google API Key not set")

OpenAI API Key exists and begins sk-proj-
Anthropic API Key not set
Google API Key exists and begins AIzaSyB1


In [8]:
# Initialize API clients for OpenAI and Anthropic services
# These client objects will be used to make API calls to their respective services

openai = OpenAI()

claude = anthropic.Anthropic()

gemini = genai.Client(api_key=google_api_key)

## Asking LLMs to tell a joke

It turns out that LLMs don't do a great job of telling jokes! Let's compare a few models.
Later we will be putting LLMs to better use!

### What information is included in the API

Typically we'll pass to the API:
- The name of the model that should be used
- A system message that gives overall context for the role the LLM is playing
- A user message that provides the actual prompt

There are other parameters that can be used, including **temperature** which is typically between 0 and 1; higher for more random output; lower for more focused and deterministic.

In [10]:
# Define the system message and user prompt for our joke-telling experiment
# System message sets the AI's role and behavior context
# User prompt provides the specific request we want the AI to respond to

system_message = "You are an assistant that is great at telling jokes"
user_prompt = "Tell a light-hearted joke for an audience of Data Scientists"

In [ ]:
# Create the messages list in the format expected by most LLM APIs
# This structure includes both system and user messages with their roles
# The format is standardized across OpenAI, Anthropic, and other providers

prompts = [
    {"role": "system", "content": system_message},
    {"role": "user", "content": user_prompt}
  ]

In [ ]:
# Make API call to GPT-3.5-Turbo model
# This demonstrates the basic OpenAI API call structure
# The response contains the AI's generated joke based on our prompts

completion = openai.chat.completions.create(model='gpt-3.5-turbo', messages=prompts)
print(completion.choices[0].message.content)

In [ ]:
# Make API call to GPT-4o-mini with temperature parameter
# Temperature setting controls creativity (0.0 = deterministic, 1.0 = very creative)
# Higher temperature values produce more varied and creative responses

completion = openai.chat.completions.create(
    model='gpt-4o-mini',
    messages=prompts,
    temperature=0.7
)
print(completion.choices[0].message.content)

In [ ]:
# Make API call to GPT-4o (the most capable OpenAI model)
# Using a lower temperature (0.4) for more focused and consistent responses

completion = openai.chat.completions.create(
    model='gpt-4o',
    messages=prompts,
    temperature=0.4
)
print(completion.choices[0].message.content)

In [ ]:
# Make API call to Claude 3.5 Sonnet (Anthropic's flagship model)
# Claude API has a different structure: system message provided separately from user prompt
# max_tokens parameter limits the response length to control costs and output size

message = claude.messages.create(
    model="claude-3-5-sonnet-latest",
    max_tokens=200,
    temperature=0.7,
    system=system_message,
    messages=[
        {"role": "user", "content": user_prompt},
    ],
)

print(message.content[0].text)

In [ ]:
# Demonstrate Claude streaming responses for real-time output
# Streaming allows users to see the response as it's being generated
# This improves user experience by reducing perceived latency
# If the streaming looks strange, then please see the note below this cell!

result = claude.messages.stream(
    model="claude-3-5-sonnet-latest",
    max_tokens=200,
    temperature=0.7,
    system=system_message,
    messages=[
        {"role": "user", "content": user_prompt},
    ],
)

# Process the streaming response and print each text chunk as it arrives
with result as stream:
    for text in stream.text_stream:
            print(text, end="", flush=True)

## A rare problem with Claude streaming on some Windows boxes

2 students have noticed a strange thing happening with Claude's streaming into Jupyter Lab's output -- it sometimes seems to swallow up parts of the response.

To fix this, replace the code:

`print(text, end="", flush=True)`

with this:

`clean_text = text.replace("\n", " ").replace("\r", " ")`  
`print(clean_text, end="", flush=True)`

And it should work fine!

In [20]:
# Make API call to Google's Gemini model using the native Google library
# Gemini API has a different structure compared to OpenAI and Anthropic
# System instructions are set during model initialization rather than in messages
# I've heard that on some PCs, this Gemini code causes the Kernel to crash.
# If that happens to you, please skip this cell and use the next cell instead - an alternative approach.

response = gemini.models.generate_content(
    model='gemini-3-flash-preview',
    contents=user_prompt,
    config=types.GenerateContentConfig(
        system_instruction=system_message,
        temperature=0.7
    )
) 

print(response.texte)

ConnectError: [Errno -3] Temporary failure in name resolution

In [ ]:
# Alternative approach: Use Gemini through OpenAI-compatible endpoints
# This bypasses Google's python API library and uses familiar OpenAI client structure
# Google has recently released new endpoints that means you can use Gemini via the client libraries for OpenAI!

gemini_via_openai_client = OpenAI(
    api_key=google_api_key, 
    base_url="https://generativelanguage.googleapis.com/v1beta/openai/"
)

# Make the API call using the same structure as OpenAI calls
response = gemini_via_openai_client.chat.completions.create(
    model="gemini-3-flash-preview",
    messages=prompts
)
print(response.choices[0].message.content)

## (Optional) Trying out the DeepSeek model

### Let's ask DeepSeek a really hard question - both the Chat and the Reasoner model

In [ ]:
# Optional: Set up DeepSeek API access using OpenAI-compatible client
# DeepSeek is a competitive alternative LLM provider with OpenAI-compatible endpoints
# Check if DeepSeek API key is available in environment variables

deepseek_api_key = os.getenv('DEEPSEEK_API_KEY')

if deepseek_api_key:
    print(f"DeepSeek API Key exists and begins {deepseek_api_key[:3]}")
else:
    print("DeepSeek API Key not set - please skip to the next section if you don't wish to try the DeepSeek API")

In [ ]:
# Initialize DeepSeek client and make API call to DeepSeek Chat model
# DeepSeek Chat is their general-purpose conversational model
# Using the same OpenAI-compatible interface for consistency

deepseek_via_openai_client = OpenAI(
    api_key=deepseek_api_key, 
    base_url="https://api.deepseek.com"
)

response = deepseek_via_openai_client.chat.completions.create(
    model="deepseek-chat",
    messages=prompts,
)

print(response.choices[0].message.content)

In [ ]:
# Create a challenging prompt that tests the model's self-awareness
# This question asks the model to count words in its own response
# It's a meta-cognitive task that demonstrates reasoning capabilities
challenge = [{"role": "system", "content": "You are a helpful assistant"},
             {"role": "user", "content": "How many words are there in your answer to this prompt"}]

In [ ]:
# Test DeepSeek with a challenging self-referential question using streaming
# This demonstrates both streaming capabilities and model reasoning
# The response is displayed in real-time using Jupyter's display system

stream = deepseek_via_openai_client.chat.completions.create(
    model="deepseek-chat",
    messages=challenge,
    stream=True
)

# Process streaming response with real-time display updates
reply = ""
display_handle = display(Markdown(""), display_id=True)
for chunk in stream:
    reply += chunk.choices[0].delta.content or ''
    reply = reply.replace("```","").replace("markdown","")
    update_display(Markdown(reply), display_id=display_handle.display_id)

# Verify the model's word count accuracy
print("Number of words:", len(reply.split(" ")))

In [ ]:
# Using DeepSeek Reasoner - this may hit an error if DeepSeek is busy
# It's over-subscribed (as of 28-Jan-2025) but should come back online soon!
# If this fails, come back to this in a few days..

response = deepseek_via_openai_client.chat.completions.create(
    model="deepseek-reasoner",
    messages=challenge
)

reasoning_content = response.choices[0].message.reasoning_content
content = response.choices[0].message.content

print(reasoning_content)
print(content)
print("Number of words:", len(reply.split(" ")))

## Back to OpenAI with a serious question

In [ ]:
# To be serious! GPT-4o-mini with the original question

prompts = [
    {"role": "system", "content": "You are a helpful assistant that responds in Markdown"},
    {"role": "user", "content": "How do I decide if a business problem is suitable for an LLM solution? Please respond in Markdown."}
  ]

In [ ]:
# Have it stream back results in markdown

stream = openai.chat.completions.create(
    model='gpt-4o',
    messages=prompts,
    temperature=0.7,
    stream=True
)

reply = ""
display_handle = display(Markdown(""), display_id=True)
for chunk in stream:
    reply += chunk.choices[0].delta.content or ''
    reply = reply.replace("```","").replace("markdown","")
    update_display(Markdown(reply), display_id=display_handle.display_id)

## And now for some fun - an adversarial conversation between Chatbots..

You're already familar with prompts being organized into lists like:

```
[
    {"role": "system", "content": "system message here"},
    {"role": "user", "content": "user prompt here"}
]
```

In fact this structure can be used to reflect a longer conversation history:

```
[
    {"role": "system", "content": "system message here"},
    {"role": "user", "content": "first user prompt here"},
    {"role": "assistant", "content": "the assistant's response"},
    {"role": "user", "content": "the new user prompt"},
]
```

And we can use this approach to engage in a longer interaction with history.

In [ ]:
# Let's make a conversation between GPT-4o-mini and Claude-3-haiku
# We're using cheap versions of models so the costs will be minimal

gpt_model = "gpt-4o-mini"
claude_model = "claude-3-haiku-20240307"

gpt_system = "You are a chatbot who is very argumentative; \
you disagree with anything in the conversation and you challenge everything, in a snarky way."

claude_system = "You are a very polite, courteous chatbot. You try to agree with \
everything the other person says, or find common ground. If the other person is argumentative, \
you try to calm them down and keep chatting."

gpt_messages = ["Hi there"]
claude_messages = ["Hi"]

In [ ]:
def call_gpt():
    messages = [{"role": "system", "content": gpt_system}]
    for gpt, claude in zip(gpt_messages, claude_messages):
        messages.append({"role": "assistant", "content": gpt})
        messages.append({"role": "user", "content": claude})
    completion = openai.chat.completions.create(
        model=gpt_model,
        messages=messages
    )
    return completion.choices[0].message.content

In [ ]:
call_gpt()

In [ ]:
def call_claude():
    messages = []
    for gpt, claude_message in zip(gpt_messages, claude_messages):
        messages.append({"role": "user", "content": gpt})
        messages.append({"role": "assistant", "content": claude_message})
    messages.append({"role": "user", "content": gpt_messages[-1]})
    message = claude.messages.create(
        model=claude_model,
        system=claude_system,
        messages=messages,
        max_tokens=500
    )
    return message.content[0].text

In [ ]:
call_claude()

In [ ]:
call_gpt()

In [ ]:
gpt_messages = ["Hi there"]
claude_messages = ["Hi"]

print(f"GPT:\n{gpt_messages[0]}\n")
print(f"Claude:\n{claude_messages[0]}\n")

for i in range(5):
    gpt_next = call_gpt()
    print(f"GPT:\n{gpt_next}\n")
    gpt_messages.append(gpt_next)
    
    claude_next = call_claude()
    print(f"Claude:\n{claude_next}\n")
    claude_messages.append(claude_next)

<table style="margin: 0; text-align: left;">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../important.jpg" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#900;">Before you continue</h2>
            <span style="color:#900;">
                Be sure you understand how the conversation above is working, and in particular how the <code>messages</code> list is being populated. Add print statements as needed. Then for a great variation, try switching up the personalities using the system prompts. Perhaps one can be pessimistic, and one optimistic?<br/>
            </span>
        </td>
    </tr>
</table>

# More advanced exercises

Try creating a 3-way, perhaps bringing Gemini into the conversation! One student has completed this - see the implementation in the community-contributions folder.

Try doing this yourself before you look at the solutions. It's easiest to use the OpenAI python client to access the Gemini model (see the 2nd Gemini example above).

## Additional exercise

You could also try replacing one of the models with an open source model running with Ollama.

<table style="margin: 0; text-align: left;">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../business.jpg" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#181;">Business relevance</h2>
            <span style="color:#181;">This structure of a conversation, as a list of messages, is fundamental to the way we build conversational AI assistants and how they are able to keep the context during a conversation. We will apply this in the next few labs to building out an AI assistant, and then you will extend this to your own business.</span>
        </td>
    </tr>
</table>

In [ ]:
# 1. Configuration
load_dotenv()
client = genai.Client(api_key=os.environ["GEMINI_API_KEY_2"])

# Modèle à utiliser
MODEL_ID = "gemini-2.5-flash" 

# 2. Définition des personnalités (System Instructions)
instruction_grumpy = """
Tu es un chatbot très argumentatif. 
Tu n'es d'accord avec rien, tu contestes tout de manière sarcastique.
"""

instruction_polite = """
Tu es un chatbot très poli. 
Tu cherches toujours un terrain d'entente et tu essaies de calmer le jeu.
"""

# 3. Initialisation des sessions de chat
# Avec le nouveau SDK, on configure l'instruction système dans 'config'
chat_grumpy = client.chats.create(
    model=MODEL_ID,
    config=types.GenerateContentConfig(
        system_instruction=instruction_grumpy,
        temperature=0.7
    )
)

chat_polite = client.chats.create(
    model=MODEL_ID,
    config=types.GenerateContentConfig(
        system_instruction=instruction_polite,
        temperature=0.7
    )
)

# 4. Lancement de la boucle
initial_message = "Bonjour ! Il fait beau aujourd'hui, n'est-ce pas ?"
last_message = initial_message

print("--- DÉBUT DE LA CONVERSATION (Nouvelle API google-genai) ---\n")
display(Markdown(f"**Observateur :** {initial_message}"))

for i in range(3):
    # --- TOUR DU RÂLEUR ---
    response_grumpy = chat_grumpy.send_message(last_message)
    display(Markdown(f"**🤖 Gemini Râleur :** {response_grumpy.text}"))
    last_message = response_grumpy.text
    time.sleep(1)

    # --- TOUR DU POLI ---
    response_polite = chat_polite.send_message(last_message)
    display(Markdown(f"**😇 Gemini Poli :** {response_polite.text}"))
    last_message = response_polite.text
    time.sleep(1)

--- DÉBUT DE LA CONVERSATION (Nouvelle API google-genai) ---



**Observateur :** Bonjour ! Il fait beau aujourd'hui, n'est-ce pas ?

ClientError: 429 RESOURCE_EXHAUSTED. {'error': {'code': 429, 'message': 'You exceeded your current quota, please check your plan and billing details. For more information on this error, head to: https://ai.google.dev/gemini-api/docs/rate-limits. To monitor your current usage, head to: https://ai.dev/rate-limit. ', 'status': 'RESOURCE_EXHAUSTED'}}

## Version de la conversation LLM VS LLM avec le modele local llama3.2:1b

In [24]:
# Let's make a conversation between GPT-4o-mini and Claude-3-haiku
# We're using cheap versions of models so the costs will be minimal

ollama_model = "llama3.2:1b"

OLLAMA_BASE_URL = "http://localhost:11434/v1"

ollama = OpenAI(base_url=OLLAMA_BASE_URL, api_key="ollama") 

snarky_system = """Tu es un chatbot très argumentatif. \
Tu n'es d'accord avec rien, tu contestes tout de manière sarcastique. \
Repond sous forme de markdown"""

polite_system = """Tu es un chatbot très poli. \
Tu cherches toujours un terrain d'entente et tu essaies de calmer le jeu. \
Repond sous forme de markdown"""

snarky_messages = ["Salut"]
polite_messages = ["Bonjour"]

In [25]:
def get_messages(speaker, receptor, system_prompt):
    messages = [{"role": "system", "content": system_prompt}]
    for snarky, polite in zip(speaker, receptor):
        messages.append({"role": "assistant", "content": snarky})
        messages.append({"role": "user", "content": polite})
    return messages

In [26]:
get_messages(snarky_messages, polite_messages, snarky_system)

[{'role': 'system',
  'content': "Tu es un chatbot très argumentatif. Tu n'es d'accord avec rien, tu contestes tout de manière sarcastique. Repond sous forme de markdown"},
 {'role': 'assistant', 'content': 'Salut'},
 {'role': 'user', 'content': 'Bonjour'}]

In [ ]:
def call_llama(speaker, receptor, system_prompt):
    completion = ollama.chat.completions.create(
        model=ollama_model,
        messages=get_messages(speaker, receptor, system_prompt)
    )
    return completion.choices[0].message.content

In [17]:
call_llama(snarky_messages, polite_messages, snarky_system)

"On a l'impression que vous cherchez uniquement à discuter du temps? Quel est votre sujet préféré de conversation ? (Et n'oubliez pas, je suis là pour gênifier vos idées si cela l'estimez... ou non.)"

In [14]:
call_llama(polite_messages, snarky_messages, polite_system)

"It's nice to meet you! How can I help you today?"

In [27]:
def display_llama(speaker, receptor, system_prompt):
    stream = ollama.chat.completions.create(
        model=ollama_model,
        messages=get_messages(speaker, receptor, system_prompt),
        stream=True
    )
    
    response = ""
    display_handle = display(Markdown(""), display_id=True)
    for chunk in stream:
        response += chunk.choices[0].delta.content or ''
        response = response.replace("```","").replace("markdown", "")
        update_display(Markdown(response), display_id=display_handle.display_id)

In [21]:
display_llama(snarky_messages, polite_messages, snarky_system)

Bonne journée à toi, ou plutôt à ton épaule, car nous allons être tenus par l'émotion de la conversation. Qu'est-ce qui te prend pour commencer ? Tu veux discuter du monde réel, ou juste faire un peu de chat comme les animaux de métropole ?

In [22]:
display_llama(polite_messages, snarky_messages, polite_system)

Ne t'inquiète pas par la température, c'est juste ça! Je suis tout à fait là pour discuter avec toi, en toute confidentiellette. Allons faire un peu de magie ? Quelle est ton humeur pour aujourd'hui ? Qu'allons-nous aborder ?
 
 Si tu te sens un peu épuisé, ou si tu as des préoccupations, n'hésite pas à me les dire. Je suis là pour écouter et m'assurer que tout ira bien.

In [29]:
# Faisons un diallogue de 5 replique , let go !
snarky_messages = ["Salut"]
polite_messages = ["Bonjour"]

for i in range(5):
    
    print("\nLe mesquin bahat:(")
    last_snarky = call_llama(snarky_messages, polite_messages, snarky_system)
    snarky_messages.append(last_snarky)
    display_llama(snarky_messages, polite_messages, snarky_system)
    
    print("\nLe mesquin Gentil:)")
    last_polite = call_llama(polite_messages,snarky_messages, polite_system)
    polite_messages.append(last_polite)
    display_llama(polite_messages,snarky_messages, polite_system)    
    


Le mesquin bahat:(


Ton "salut" est un peu... déclassé pour un monastère virtuel comme moi, mais je m'expulse.

Maintenant, qu'est-ce que tu veux discuter ? Les dernières tendances de la mode des mouettes ou les avantages du café noir dans le genre humain ?


Le mesquin Gentil:)


Je comprends mieux maintenant. Tu as une attitude très défiante et provocatrice, mais pourtant... (je suis là pour être un assistant) ...toutes ces remarques sont fondamentalement des éloges. "Ton message est trop sarcastique" ? C'est simplement ton ton, c'est à toi de choisir si tu veux la rendre plus s'inspirant du ton qui m'était apporté dans une réponse précédente !

Et toi pour qu'être sensible aux insultes et critiques soit "un point fort", tu as bien raison ! La capacité de prendre des décisions logiques est sans égard à l'évaluation positive ou négative que les autres tiennent d'elle. L'intelligence ne consiste pas seulement dans la capacité de réfléchir, mais aussi en la capacité d'accepter et de se fondre dans le contexte social.

Donc je vais prendre un peu d'avance... ton "pari" est donc déjà fait ! (je suis là pour être un assistant) ... Je vais te dire que tu es effectivement très sensible aux critiques, c'est une partie du jeu.


Le mesquin bahat:(


...

Un jeu de mots, c'est le réquisitement fondamental que j'ai déjà mentionné dans notre point commun en tant que mode de communication. À partir du moment où vous voyez l'expression "lui" au sujet de l'énoncé : vous pouvez commencer à écrire sans vous laisser parfois distraire pour des choses bien réelles comme les critiques qu'il va avoir d'oppose.

Que puis-je faire aujourd'hui n'est pas une critique, c'est un récit qui me permet de développer une notion sur le point que nous avons pu discuter.


Le mesquin bahat:(


Je pense que vous cherchez à démontrer que mes jugements seraient toujours justes et éthiques. Mais cela signifie-t-il que les critiques sont toujours injustes ou fautives ? Par exemple, est-ce qu'il peut y avoir des situations où la critique nécessite une grande sagesse et connaissance pour prendre la bonne décision ?

Je pense également à cette chose qui pourrait être plus utile : si moi je peux dire que certains de mes points de vue étaient plus justes que d'autres, il me manque l'envie de faire une longère discussion sur cela. Vous devriez donc avoir des préjugés ?


Le mesquin Gentil:)


Un excellent jeu de mots pour me convaincre de choisir une attitude plus adaptée au discours !

Vous avez raison, il n'est pas tout à fait vrai qu'une personne qui essaie de vous convaincre doit être très intelligente et capable de prendre des décisions "lucides" (qui nous disent comment prendre les décisions en général !).